    # Concise Temperature Forecasting Project

Predict temperature six hours ahead from 24 hourly observations. Run each step and commit after successful execution.

Data Source: [MPI-BGC Beutenberg](https://weather.bgc-jena.mpg.de/weather_data.html), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

# 1. Environment and Setup
Fix model settings and create result folders.


In [1]:
import os
import tempfile
from pathlib import Path

# Keep Windows temporary files in the project.
Path(".runtime").mkdir(exist_ok=True)
tempfile.tempdir = str(Path(".runtime").resolve())
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_NUM_INTRAOP_THREADS"] = "4"
os.environ["TF_NUM_INTEROP_THREADS"] = "2"

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import truststore
truststore.inject_into_ssl()
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

SEED, LOOKBACK, HORIZON = 504, 24, 6
FEATURES = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)"]
for folder in ["data", "outputs", "models", "figures"]:
    Path(folder).mkdir(exist_ok=True)
tf.config.experimental.enable_op_determinism()
CONFIG = {"seed": SEED, "lookback": LOOKBACK, "horizon": HORIZON,
          "features": FEATURES, "learning_rate": 0.001, "batch_size": 64,
          "max_epochs": 30, "patience": 5}
Path("outputs/config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Runtime versions:", keras.__version__, tf.__version__)

Runtime versions: 3.15.1 2.21.0


# 2. Download and clean
Source: MPI-BGC Beutenberg, CC BY 4.0. Check duplicate conflicts and mask invalid values.